In [30]:
# import a utility function for loading Roboflow models
from inference import get_model

# define the local image path to use for inference
image = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/roboflow_dataset_2/train/images/country1_000053_jpg.rf.3cbf340cc0f618ff3a4932e3f39e301a.jpg"

# load the road damage detection model
model = get_model(model_id="road-damage-5lxtz-zqhl5/5", api_key="IJU8d9jXU38Ts1Fqoev2")

# run inference on our chosen image, image can be a url, a numpy array, a PIL image, etc.
results = model.infer(image)

In [31]:
results[0].predictions

[ObjectDetectionPrediction(x=439.0, y=536.0, width=402.0, height=204.0, confidence=0.7438175678253174, class_name='1', class_confidence=None, class_id=1, tracker_id=None, detection_id='edb50df5-41c0-4186-beae-f8d4f9b1b821', parent_id=None),
 ObjectDetectionPrediction(x=356.0, y=525.5, width=50.0, height=209.0, confidence=0.5141727924346924, class_name='3', class_confidence=None, class_id=3, tracker_id=None, detection_id='b9a8953d-8a24-429d-ac80-051d07c4acf0', parent_id=None)]

In [32]:
results[0].image.width, results[0].image.height

(640, 640)

In [33]:
def convert_results_to_yolo_format(results):
    """
    Convert Roboflow inference results to YOLO format string.
    
    Args:
        results: Roboflow inference results object
        
    Returns:
        str: YOLO format string with format:
             <class_id> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>
    """
    if not results or len(results) == 0:
        return ""
    
    # Get image dimensions
    img_width = results[0].image.width
    img_height = results[0].image.height
    
    yolo_lines = []
    
    # Process each prediction
    for prediction in results[0].predictions:
        # Extract values
        class_id = prediction.class_id
        x_center = prediction.x
        y_center = prediction.y
        width = prediction.width
        height = prediction.height
        confidence = prediction.confidence
        
        # Normalize coordinates (convert to 0-1 range)
        x_center_normalized = x_center / img_width
        y_center_normalized = y_center / img_height
        width_normalized = width / img_width
        height_normalized = height / img_height
        
        # Format as YOLO string
        yolo_line = f"{class_id} {x_center_normalized:.6f} {y_center_normalized:.6f} {width_normalized:.6f} {height_normalized:.6f} {confidence:.6f}"
        yolo_lines.append(yolo_line)
    
    return "\n".join(yolo_lines)

# Test the function with the current results
yolo_format_string = convert_results_to_yolo_format(results)
print("YOLO Format Output:")
print(yolo_format_string)

YOLO Format Output:
1 0.685937 0.837500 0.628125 0.318750 0.743818
3 0.556250 0.821094 0.078125 0.326562 0.514173


In [ ]:
import os
import glob
from pathlib import Path
from tqdm import tqdm

# Configuration
TEST_DATA_DIR = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/test_data"
OUTPUT_DIR = "/home/kwdahun/2025IMSC-Hackathon-CRACKPINK/inference_result_4"

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

def get_all_test_images():
    """Collect all test images from all countries"""
    all_images = []
    
    for country in ['country_1', 'country_2', 'country_3']:
        country_path = os.path.join(TEST_DATA_DIR, country, 'images')
        if os.path.exists(country_path):
            image_files = glob.glob(os.path.join(country_path, '*.jpg'))
            for img_path in image_files:
                all_images.append({
                    'path': img_path,
                    'country': country,
                    'filename': os.path.basename(img_path)
                })
    
    return all_images

def process_image_and_save(image_info, model, output_dir):
    """
    Process a single image with Roboflow model and save results in YOLO format
    """
    image_path = image_info['path']
    filename = image_info['filename']
    
    try:
        # Run inference
        results = model.infer(image_path, overlap=15)
        
        # Convert to YOLO format
        yolo_string = convert_results_to_yolo_format(results)
        
        # Prepare output txt file path (same name as image but with .txt extension)
        txt_filename = os.path.splitext(filename)[0] + '.txt'
        output_txt_path = os.path.join(output_dir, txt_filename)
        
        # Save to file
        with open(output_txt_path, 'w') as f:
            f.write(yolo_string)
        
        # Count detections
        num_detections = len(results[0].predictions) if results and len(results) > 0 else 0
        
        return {
            'image_path': image_path,
            'output_path': output_txt_path,
            'num_detections': num_detections,
            'success': True
        }
        
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return {
            'image_path': image_path,
            'error': str(e),
            'success': False
        }

# Get all test images
test_images = get_all_test_images()
print(f"Total test images found: {len(test_images)}")

# Show distribution by country
for country in ['country_1', 'country_2', 'country_3']:
    country_count = len([img for img in test_images if img['country'] == country])
    print(f"{country}: {country_count} images")

Output directory: /home/kwdahun/2025IMSC-Hackathon-CRACKPINK/inference_result_4
Total test images found: 2961
country_1: 1024 images
country_2: 918 images
country_3: 1019 images


In [35]:
# Run batch processing on all test images
print("Starting batch inference on all test images...")
print(f"Processing {len(test_images)} images...")

inference_results = []
failed_images = []

# Process images with progress bar
for i, image_info in enumerate(tqdm(test_images, desc="Processing images")):
    result = process_image_and_save(image_info, model, OUTPUT_DIR)
    
    if result['success']:
        inference_results.append(result)
    else:
        failed_images.append(result)
    
    # Print progress every 100 images
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(test_images)} images...")

print(f"\nBatch inference completed!")
print(f"Successfully processed: {len(inference_results)} images")
print(f"Failed to process: {len(failed_images)} images")

# Print summary statistics
total_detections = sum(result['num_detections'] for result in inference_results)
print(f"Total detections found: {total_detections}")
print(f"Average detections per image: {total_detections / len(inference_results):.2f}")

# Show some examples of processed files
print("\nFirst 5 processed files:")
for i, result in enumerate(inference_results[:5]):
    print(f"{i+1}. {os.path.basename(result['output_path'])} - {result['num_detections']} detections")

Starting batch inference on all test images...
Processing 2961 images...


Processing images:   4%|▎         | 108/2961 [00:01<00:47, 60.28it/s]

Processed 100/2961 images...


Processing images:   7%|▋         | 210/2961 [00:03<00:46, 59.60it/s]

Processed 200/2961 images...


...

Processed 2800/2961 images...


Processing images:  98%|█████████▊| 2909/2961 [00:44<00:00, 74.80it/s]

Processed 2900/2961 images...


Processing images: 100%|██████████| 2961/2961 [00:45<00:00, 65.08it/s]


Batch inference completed!
Successfully processed: 2961 images
Failed to process: 0 images
Total detections found: 4814
Average detections per image: 1.63

First 5 processed files:
1. country1_008643.txt - 0 detections
2. country1_002089.txt - 1 detections
3. country1_007890.txt - 1 detections
4. country1_007682.txt - 1 detections
5. country1_008386.txt - 2 detections


In [23]:
# Check current progress and verify output format
import os

# Check how many files were created
existing_files = glob.glob(os.path.join(OUTPUT_DIR, "*.txt"))
print(f"Number of txt files created so far: {len(existing_files)}")

# Display content of a few sample files to verify format
if existing_files:
    print("\nSample output files:")
    for i, file_path in enumerate(existing_files[:3]):
        filename = os.path.basename(file_path)
        print(f"\n{filename}:")
        with open(file_path, 'r') as f:
            content = f.read().strip()
            if content:
                print(content)
            else:
                print("(empty file - no detections)")

# Show the expected format
print("\nExpected format:")
print("<class_id> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>")
print("Example: 1 0.681293 0.833369 0.620362 0.331212 0.883995")

Number of txt files created so far: 2961

Sample output files:

country3_001614.txt:
3 0.407813 0.746875 0.090625 0.221875 0.805841
2 0.581250 0.848437 0.175000 0.034375 0.667629
2 0.092188 0.749219 0.184375 0.039062 0.639264
2 0.427344 0.639844 0.195312 0.026562 0.444208

country2_007061.txt:
1 0.671667 0.845833 0.656667 0.308333 0.756931
0 0.340000 0.860833 0.093333 0.048333 0.555188
2 0.868333 0.929167 0.263333 0.048333 0.478949

country2_006275.txt:
1 0.772500 0.331667 0.448333 0.636667 0.684108
2 0.428333 0.130000 0.300000 0.046667 0.529701
2 0.600833 0.304167 0.188333 0.051667 0.520252

Expected format:
<class_id> <x_center_normalized> <y_center_normalized> <width_normalized> <height_normalized> <confidence>
Example: 1 0.681293 0.833369 0.620362 0.331212 0.883995


In [16]:
# Resumable batch processing - skip already processed images
def get_remaining_images(all_images, output_dir):
    """Get list of images that haven't been processed yet"""
    existing_txt_files = set()
    for txt_file in glob.glob(os.path.join(output_dir, "*.txt")):
        base_name = os.path.splitext(os.path.basename(txt_file))[0]
        existing_txt_files.add(base_name)
    
    remaining_images = []
    for img_info in all_images:
        img_base_name = os.path.splitext(img_info['filename'])[0]
        if img_base_name not in existing_txt_files:
            remaining_images.append(img_info)
    
    return remaining_images

# Get remaining images to process
remaining_images = get_remaining_images(test_images, OUTPUT_DIR)
print(f"Already processed: {len(test_images) - len(remaining_images)} images")
print(f"Remaining to process: {len(remaining_images)} images")

if len(remaining_images) > 0:
    print("\nTo continue processing, run the batch processing loop with remaining_images instead of test_images")
    print("You can also process in smaller batches to avoid interruption")
else:
    print("All images have been processed!")

Already processed: 2961 images
Remaining to process: 0 images
All images have been processed!
